[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almahova/iemocap-proj/blob/data-modelling/iemocap_modelling.ipynb)

# IEMOCAP Emotion Recognition Modelling Pipeline
## Intermediate Fusion

This notebook implements a **Text-and-Audio Transformer pipeline** for 3-class Speech Emotion Recognition (SER) on the IEMOCAP corpus.

| Component | Choice |
|-----------|--------|
| Model | `roberta-base` + `wav2vec2-base` encoders with shared MLP fusion head |
| Split | Sessions 1–4 train · 10 % of train as val · Session 5 test (speaker-independent) |
| Loss | Weighted Cross-Entropy (compensates ~55 % Negative imbalance) |
| Primary metric | **Macro F1-Score** |
| Classes | `Negative` (0) · `Positive` (1) · `Neutral` (2) |

## Implementation Guidelines

Using pre-trained RoBERTa and Wav2Vec2 models to classify speech into 3 emotion groups:
**Neutral**, **Positive**, and **Negative**.

- Split: **Sessions 1–4 train / 10 % of train as validation / Session 5 held-out test** (speaker-independent)
- Address class imbalance with **inverse-frequency class weights** in the loss function
- Report **confusion matrix**, **accuracy**, and **Macro F1-Score**
- Use **early stopping** to prevent overfitting (patience = 3 epochs on val Macro F1)

---
## 1. Configuration & Imports

In [ ]:
# ── All hyperparameters in one place change here, nowhere else ──────────────
CONFIG = {
    "text_model_name":   "roberta-base",
    "audio_model_name":  "facebook/wav2vec2-base",
    "text_column":       "transcription",
    "max_length":        128,     # text tokenizer max length
    "max_duration":      4.0,     # audio seconds must match audio_best_model.pt training
    "target_sr":         16_000,  # Hz required by Wav2Vec2
    "batch_size":        8,       # joint model is heavier than single-modality
    "lr_encoder":        1e-5,    # lower LR for the pretrained encoders
    "lr_head":           1e-4,    # higher LR for the new fusion head
    "weight_decay":      0.01,
    "epochs":            10,
    "early_stop_patience": 3,
    "test_session":      5,       # Ses05 held out as test set
    "val_ratio":         0.10,    # carved from Sessions 1-4 for early stopping
    "random_seed":       42,
    "num_labels":        3,
    "fusion_hidden_dim": 256,     # hidden size of the fusion MLP head
    "freeze_audio_feature_encoder": True,
}

LABEL2ID = {"Negative": 0, "Positive": 1, "Neutral": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = ["Negative", "Positive", "Neutral"]

EMOTION_GROUP_MAP = {
    "happy":      "Positive",
    "excited":    "Positive",
    "angry":      "Negative",
    "frustrated": "Negative",
    "fear":       "Negative",
    "sad":        "Negative",
    "disgust":    "Negative",
    "other":      "Neutral",
    "neutral":    "Neutral",
    "surprise":   "Neutral",
}

import os
import io
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import torchaudio.transforms as T
import warnings
warnings.filterwarnings("ignore")

from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaForSequenceClassification,
    Wav2Vec2Processor,
    Wav2Vec2Model,
    Wav2Vec2ForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, accuracy_score,
    confusion_matrix, classification_report,
)
from tqdm import tqdm

torch.manual_seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

# ── Device — CUDA > MPS (Apple Silicon) > CPU ────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

# ── Checkpoint locations — Google Drive in Colab, ~/Downloads locally ─────────
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = "/content/drive/MyDrive"
else:
    CHECKPOINT_DIR = os.path.expanduser("~/Downloads")

TEXT_CKPT_PATH  = os.path.join(CHECKPOINT_DIR, "text_best_model_weights.pt")
AUDIO_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "audio_best_model.pt")

---
## 2. Data Loading & Preprocessing

In [ ]:
from datasets import load_dataset


def load_and_preprocess(
    emotion_map: dict,
    label2id: dict,
    text_col: str,
) -> pd.DataFrame:
    """Load IEMOCAP from HuggingFace, group emotions, encode labels, drop unknowns.

    Returns a DataFrame with columns: text_col, audio, file, emotion_group, label, session_num.
    """
    ds = load_dataset("AbstractTTS/IEMOCAP")
    df = ds["train"].to_pandas()

    df["emotion_group"] = df["major_emotion"].map(emotion_map)
    df = df.dropna(subset=["emotion_group", text_col, "audio"]).reset_index(drop=True)
    df["label"] = df["emotion_group"].map(label2id)

    # Extract integer session number (1-5) for LOSO grouping
    df["session_num"] = df["file"].str.extract(r"Ses(\d{2})").astype(int)

    print(f"Total samples after cleaning: {len(df)}")
    print(df["emotion_group"].value_counts().to_string())
    return df


def loso_split(
    df: pd.DataFrame,
    test_session: int,
    val_ratio: float,
    seed: int,
) -> tuple:
    """LOSO split: `test_session` -> test; remaining sessions -> stratified train/val.

    Returns: (train_df, val_df, test_df)
    """
    test_df     = df[df["session_num"] == test_session].reset_index(drop=True)
    trainval_df = df[df["session_num"] != test_session].reset_index(drop=True)

    train_df, val_df = train_test_split(
        trainval_df,
        test_size=val_ratio,
        stratify=trainval_df["label"],
        random_state=seed,
    )

    for name, part in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        counts = " | ".join(
            f"{k}: {v}" for k, v in part["emotion_group"].value_counts().items()
        )
        print(f"{name:5s}: {len(part):5d} rows | {counts}")

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


df = load_and_preprocess(EMOTION_GROUP_MAP, LABEL2ID, CONFIG["text_column"])
train_df, val_df, test_df = loso_split(
    df, CONFIG["test_session"], CONFIG["val_ratio"], CONFIG["random_seed"]
)

---
## 3. Dataset & DataLoaders

`IEMOCAPFusionDataset` returns tokenized text **and** processed audio for each sample. Text is pre-tokenized up front (cheap); audio is loaded/resampled/processed lazily in `__getitem__` (matches `audio_classification.ipynb`'s `IEMOCAPDataset`). The two attention masks are renamed (`attention_mask_text` / `attention_mask_audio`) to avoid key collisions in the batch dict.

In [ ]:
class IEMOCAPFusionDataset(Dataset):
    """Yields tokenized text + processed audio + label for each sample."""

    def __init__(self, df, tokenizer, processor, max_length, max_duration, target_sr):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.audio_max_length = int(max_duration * target_sr)
        self.target_sr = target_sr

        # Pre-tokenize all text up front (like IEMOCAPTextDataset)
        self.text_encodings = tokenizer(
            self.df["transcription"].tolist(),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(self.df["label"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def _load_audio(self, audio_data):
        raw_bytes = audio_data.get("bytes")
        waveform, sr = sf.read(io.BytesIO(raw_bytes), dtype="float32")
        if waveform.ndim > 1:
            waveform = waveform.mean(axis=-1)
        return waveform, sr

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sr = self._load_audio(row["audio"])

        if sr != self.target_sr:
            waveform_t = torch.from_numpy(waveform).unsqueeze(0)
            resampler  = T.Resample(orig_freq=sr, new_freq=self.target_sr)
            waveform   = resampler(waveform_t).squeeze(0).numpy()

        encoded = self.processor(
            waveform,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            max_length=self.audio_max_length,
            truncation=True,
            padding="max_length",
        )

        return {
            "input_ids":            self.text_encodings["input_ids"][idx],
            "attention_mask_text":  self.text_encodings["attention_mask"][idx],
            "input_values":         encoded["input_values"].squeeze(0),
            "attention_mask_audio": encoded.get(
                "attention_mask", torch.ones(self.audio_max_length)
            ).squeeze(0),
            "labels": self.labels[idx],
        }


text_tokenizer  = RobertaTokenizer.from_pretrained(CONFIG["text_model_name"])
audio_processor = Wav2Vec2Processor.from_pretrained(CONFIG["audio_model_name"])

train_ds = IEMOCAPFusionDataset(
    train_df, text_tokenizer, audio_processor,
    CONFIG["max_length"], CONFIG["max_duration"], CONFIG["target_sr"],
)
val_ds = IEMOCAPFusionDataset(
    val_df, text_tokenizer, audio_processor,
    CONFIG["max_length"], CONFIG["max_duration"], CONFIG["target_sr"],
)
test_ds = IEMOCAPFusionDataset(
    test_df, text_tokenizer, audio_processor,
    CONFIG["max_length"], CONFIG["max_duration"], CONFIG["target_sr"],
)

# num_workers=0 required on macOS — multiprocessing spawn can't pickle notebook-defined classes
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0, pin_memory=False)

print(f"Batches train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

---
## 4. Model Architecture

### Transfer learning from the existing fine-tuned checkpoints

`load_text_encoder` / `load_audio_encoder` load the **encoder-only** weights from the full classification checkpoints (`text_best_model_weights.pt` -> `roberta.*` keys, `best_model.pt` -> `wav2vec2.*` keys), discarding the old single-modality classification heads. `strict=False` is used because the bare encoder modules (`RobertaModel`, `Wav2Vec2Model`) may not have an exact 1:1 key match (e.g. pooler/`position_ids` buffers) check the printed missing/unexpected counts on first run.

### Pooling & fusion

- **Text branch**: mean-pool `last_hidden_state` over non-padded tokens (using `attention_mask_text`).
- **Audio branch**: mean-pool `last_hidden_state` over the time axis (`audio_hidden.mean(dim=1)`) mirrors the pooling `Wav2Vec2ForSequenceClassification` applies before its `projector`/`classifier`. A properly downsampled attention-mask-aware mean pool is a possible future improvement, not implemented here.

The two pooled vectors (768 + 768 = 1536-dim) are concatenated and fed through a small MLP fusion head -> 3 logits.

In [ ]:
def load_text_encoder(model_name, ckpt_path, device):
    """Load a RoBERTa base encoder, initialised from a fine-tuned classification checkpoint."""
    full_sd = torch.load(ckpt_path, map_location="cpu")
    # Keys look like 'roberta.embeddings...', 'roberta.encoder...', 'classifier.*'
    encoder_sd = {
        k[len("roberta."):]: v
        for k, v in full_sd.items()
        if k.startswith("roberta.")
    }
    encoder = RobertaModel.from_pretrained(model_name, add_pooling_layer=False)
    missing, unexpected = encoder.load_state_dict(encoder_sd, strict=False)
    print(f"[text encoder]  missing={len(missing)} unexpected={len(unexpected)}")
    return encoder


def load_audio_encoder(model_name, ckpt_path, device, freeze_feature_encoder=True):
    """Load a Wav2Vec2 base encoder, initialised from a fine-tuned classification checkpoint."""
    full_sd = torch.load(ckpt_path, map_location="cpu")
    # Keys look like 'wav2vec2.feature_extractor...', 'projector.*', 'classifier.*'
    encoder_sd = {
        k[len("wav2vec2."):]: v
        for k, v in full_sd.items()
        if k.startswith("wav2vec2.")
    }
    encoder = Wav2Vec2Model.from_pretrained(model_name)
    missing, unexpected = encoder.load_state_dict(encoder_sd, strict=False)
    print(f"[audio encoder] missing={len(missing)} unexpected={len(unexpected)}")
    if freeze_feature_encoder:
        encoder.feature_extractor._freeze_parameters()
    return encoder


class IntermediateFusionModel(nn.Module):
    def __init__(self, text_encoder, audio_encoder, num_labels, fusion_hidden_dim, dropout=0.3):
        super().__init__()
        self.text_encoder = text_encoder
        self.audio_encoder = audio_encoder

        text_dim  = text_encoder.config.hidden_size   # 768 for roberta-base
        audio_dim = audio_encoder.config.hidden_size  # 768 for wav2vec2-base
        fused_dim = text_dim + audio_dim

        self.fusion_head = nn.Sequential(
            nn.Linear(fused_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, num_labels),
        )

    def forward(self, input_ids, attention_mask_text, input_values, attention_mask_audio):
        # --- Text branch: mean-pool last hidden state over non-padded tokens ---
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask_text)
        text_hidden = text_out.last_hidden_state                  # (B, T_text, 768)
        text_mask = attention_mask_text.unsqueeze(-1).float()     # (B, T_text, 1)
        text_pooled = (text_hidden * text_mask).sum(1) / text_mask.sum(1).clamp(min=1e-9)

        # --- Audio branch: mean-pool over time ---
        audio_out = self.audio_encoder(input_values=input_values, attention_mask=attention_mask_audio)
        audio_hidden = audio_out.last_hidden_state                # (B, T_audio, 768)
        audio_pooled = audio_hidden.mean(dim=1)

        fused = torch.cat([text_pooled, audio_pooled], dim=-1)
        logits = self.fusion_head(fused)
        return logits


text_encoder  = load_text_encoder(CONFIG["text_model_name"], TEXT_CKPT_PATH, device)
audio_encoder = load_audio_encoder(
    CONFIG["audio_model_name"], AUDIO_CKPT_PATH, device,
    freeze_feature_encoder=CONFIG["freeze_audio_feature_encoder"],
)

model = IntermediateFusionModel(
    text_encoder, audio_encoder,
    num_labels=CONFIG["num_labels"],
    fusion_hidden_dim=CONFIG["fusion_hidden_dim"],
).to(device)

print(model)

---
## 5. Class Weights & Optimiser

In [ ]:
train_labels = train_df["label"].values
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1, 2]), y=train_labels
)
weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

print("Class weights:", {ID2LABEL[i]: f"{w:.3f}" for i, w in enumerate(class_weights)})

# Differential learning rates: lower for the pretrained encoders, higher for the new fusion head
encoder_params = list(model.text_encoder.parameters()) + list(model.audio_encoder.parameters())
head_params    = list(model.fusion_head.parameters())

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": CONFIG["lr_encoder"]},
    {"params": head_params,    "lr": CONFIG["lr_head"]},
], weight_decay=CONFIG["weight_decay"])

total_steps  = len(train_loader) * CONFIG["epochs"]
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps,
)
print(f"Total optimisation steps: {total_steps}")

---
## 6. Training Loop with Early Stopping

In [ ]:
def run_epoch(model, loader, criterion, device, optimizer=None, scheduler=None):
    """Run one epoch in train mode (if optimizer given) or eval mode. Returns (avg_loss, macro_f1)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, desc="train" if is_train else "eval", leave=False):
            input_ids      = batch["input_ids"].to(device)
            attn_text      = batch["attention_mask_text"].to(device)
            input_values   = batch["input_values"].to(device)
            attn_audio     = batch["attention_mask_audio"].to(device)
            labels         = batch["labels"].to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(input_ids, attn_text, input_values, attn_audio)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, macro_f1


history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}
best_val_f1 = 0.0
epochs_no_improve = 0
best_state = None

for epoch in range(CONFIG["epochs"]):
    tr_loss, tr_f1 = run_epoch(model, train_loader, criterion, device, optimizer, scheduler)
    vl_loss, vl_f1 = run_epoch(model, val_loader, criterion, device)

    history["train_loss"].append(tr_loss); history["train_f1"].append(tr_f1)
    history["val_loss"].append(vl_loss);   history["val_f1"].append(vl_f1)

    print(f"Epoch {epoch + 1}/{CONFIG['epochs']} | "
          f"Train loss {tr_loss:.4f} F1 {tr_f1:.4f} | Val loss {vl_loss:.4f} F1 {vl_f1:.4f}")

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        print(f"  >> New best (Val Macro F1 = {best_val_f1:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= CONFIG["early_stop_patience"]:
            print(f"Early stopping at epoch {epoch + 1}")
            break

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
print(f"\nBest val Macro F1: {best_val_f1:.4f} best checkpoint restored.")

ckpt_save_path = os.path.join(CHECKPOINT_DIR, "intermediate_fusion_best_model.pt")
torch.save(model.state_dict(), ckpt_save_path)
print(f"Saved {ckpt_save_path}")

---
## 7. Final Evaluation on Test Set

In [ ]:
test_loss, test_f1 = run_epoch(model, test_loader, criterion, device)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="test"):
        input_ids    = batch["input_ids"].to(device)
        attn_text    = batch["attention_mask_text"].to(device)
        input_values = batch["input_values"].to(device)
        attn_audio   = batch["attention_mask_audio"].to(device)
        labels       = batch["labels"]

        logits = model(input_ids, attn_text, input_values, attn_audio)
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(labels.tolist())

acc         = accuracy_score(all_labels, all_preds)
macro_f1    = f1_score(all_labels, all_preds, average="macro")
weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"Test Accuracy:    {acc:.4f}")
print(f"Test Macro F1:    {macro_f1:.4f}")
print(f"Test Weighted F1: {weighted_f1:.4f}")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Intermediate Fusion Ses{CONFIG['test_session']:02d} Test Set")
plt.tight_layout()
plt.show()

---
## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="Train")
axes[0].plot(epochs_range, history["val_loss"],   label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_f1"], label="Train")
axes[1].plot(epochs_range, history["val_f1"],   label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Macro F1")
axes[1].legend()

plt.suptitle("Intermediate Fusion Training History", y=1.02)
plt.tight_layout()
plt.show()